# LLM Zoomcamp HW2 · Vector Search

行动清单：[hw2-today.md](../learning-plan-2026/hw2-today.md)

按 Phase 顺序跑。Kernel 选 `llm-zoomcamp-hw2/.venv`。

## Phase 0 · Embedder

In [2]:
from embedder import Embedder

embed = Embedder()
print("Embedder ready")

Embedder ready


## Phase 1 · Q1

In [3]:
query_q1 = "How does approximate nearest neighbor search work?"
v = embed.encode(query_q1)
len(v)     # 384
print(v[0])

-0.020582036807885073


## Phase 2 · 加载数据 + Q2

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [f.parse() for f in reader.read()]
len(documents)

72

In [5]:
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
page = next(d for d in documents if d["filename"] == target)

page_vec = embed.encode(page["content"])
float(v.dot(page_vec))   # → Q2 答案

0.361070280302606

## Phase 3 · Q3 Chunk + 手写向量搜索

In [6]:
from gitsource import chunk_documents
import numpy as np
from tqdm.auto import tqdm

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [7]:
texts = [c["content"] for c in chunks]
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    vectors.extend(embed.encode_batch(batch))

X = np.array(vectors)
X.shape

  0%|          | 0/6 [00:00<?, ?it/s]

(295, 384)

In [8]:
scores = X.dot(v)
best_idx = int(np.argmax(scores))
chunks[best_idx]["filename"]   # → Q3 答案

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Phase 4 · Q4 VectorSearch

In [9]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

q4 = "What metric do we use to evaluate a search engine?"
q4_vec = embed.encode(q4)

results = vindex.search(q4_vec, num_results=5)
results[0]["filename"]   # → Q4 答案

'04-evaluation/lessons/05-search-metrics.md'

## Phase 5 · Q5 文本 vs 向量

In [10]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)

q5 = "How do I store vectors in PostgreSQL?"
q5_vec = embed.encode(q5)

vec_results = vindex.search(q5_vec, num_results=5)
text_results = text_index.search(q5, num_results=5)

vec_files = {r["filename"] for r in vec_results}
text_files = {r["filename"] for r in text_results}

vec_files - text_files   # → Q5 答案

{'02-vector-search/lessons/08-pgvector.md'}

## Phase 6 · Q6 Hybrid RRF

In [11]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [12]:
q6 = "How do I give the model access to tools?"
q6_vec = embed.encode(q6)

vec6 = vindex.search(q6_vec, num_results=5)
text6 = text_index.search(q6, num_results=5)

hybrid = rrf([text6, vec6])
hybrid[0]["filename"]   # → Q6 答案

# 对比：单独两种搜索的第一名
print("text #1:", text6[0]["filename"])
print("vector #1:", vec6[0]["filename"])

text #1: 01-agentic-rag/lessons/14-agentic-loop.md
vector #1: 01-agentic-rag/lessons/01-intro.md
